In [ ]:
import os
from torchvision import datasets
import requests
import zipfile
import shutil
from pathlib import Path
from dataPath import DataPath
from sklearn.model_selection import train_test_split

In [ ]:
def load_cifar_10(root_path=DataPath.CIFAR_10, overwrite=False):
    if os.path.exists(root_path):
        if not overwrite:
            print("Directory already exists and overwrite is set to false. Exiting...")
            return
        else:
            shutil.rmtree(root_path)
    
    train_set = datasets.CIFAR10(root=root_path, train=True, download=True)
    test_set = datasets.CIFAR10(root=root_path, train=False, download=True)
    
    classes = train_set.classes 

    for name, dataset in [("Train_original", train_set), ("Test_original", test_set)]:
        print(f"Processing {name} set...")
        
        for i, (img, target) in enumerate(dataset): # type: ignore
            class_name = classes[target]
            
            target_dir = os.path.join(root_path, name, class_name)
            os.makedirs(target_dir, exist_ok=True)
            
            img.save(os.path.join(target_dir, f"{i:05d}.jpg"))
    print(f"Done!")
    

In [ ]:
def load_cifar_100(root_path=DataPath.CIFAR_100, overwrite=False):
    if os.path.exists(root_path):
        if not overwrite:
            print("Directory already exists and overwrite is set to false. Exiting...")
            return
        else:
            shutil.rmtree(root_path)
    train_set = datasets.CIFAR100(root=root_path, train=True, download=True)
    test_set = datasets.CIFAR100(root=root_path, train=False, download=True)
    
    classes = train_set.classes 

    for name, dataset in [("Train_original", train_set), ("Test_original", test_set)]:
        print(f"Processing {name} set...")
        
        for i, (img, target) in enumerate(dataset): # type: ignore
            class_name = classes[target]
            
            target_dir = os.path.join(root_path, name, class_name)
            os.makedirs(target_dir, exist_ok=True)
            
            img.save(os.path.join(target_dir, f"{i:05d}.jpg"))
        
    print(f"Done!")
    

In [ ]:
def load_stl10(root_path=DataPath.STL, overwrite=False):
    if os.path.exists(root_path):
        if not overwrite:
            print("Directory already exists and overwrite is set to false. Exiting...")
            return
        else:
            shutil.rmtree(root_path)
    for split in ["train", "test"]:
        dataset = datasets.STL10(root=root_path, split=split, download=True)
        classes = dataset.classes
        
        print(f"Processing STL-10 {split}...")
        for i, (img, target) in enumerate(dataset): # type: ignore
            class_name = classes[target]
            target_dir = os.path.join(root_path, f"{split.capitalize()}_original", class_name)
            os.makedirs(target_dir, exist_ok=True)
            img.save(os.path.join(target_dir, f"{i:05d}.jpg"))
    print(f"Done!")
    

In [ ]:
def load_mnist(root_path=DataPath.MNIST, overwrite=False):
    if os.path.exists(root_path):
        if not overwrite:
            print("Directory already exists and overwrite is set to false. Exiting...")
            return
        else:
            shutil.rmtree(root_path)
            
    root = Path(root_path)
    
    train_set = datasets.MNIST(root=str(root/"temp"), train=True, download=True)
    test_set = datasets.MNIST(root=str(root/"temp"), train=False, download=True)

    for name, dataset in [("Train_original", train_set), ("Test_original", test_set)]:
        print(f"Converting MNIST {name} to image folders...")
        
        for i, (img, target) in enumerate(dataset): # type: ignore
            class_dir = root / name / str(target)
            class_dir.mkdir(parents=True, exist_ok=True)
            img.save(class_dir / f"{i}.jpg")

    if (root / "temp").exists():
        shutil.rmtree(root / "temp")
        
    print(f"Done!")

In [ ]:
load_cifar_10()

In [ ]:
load_cifar_100()

In [ ]:
load_stl10()

In [ ]:
load_mnist()

In [ ]:
def select_subset_train(data_path: DataPath, num_samples: int = 1000, seed: int = 0):    
    train_dir = Path(data_path) / "Train_original"
    new_train_dir = Path(data_path) / "Train"

    if new_train_dir.exists():
        print(f"Train already exists at {new_train_dir}. Skipping...")
        return

    all_filepaths = []
    all_labels = []
    classes = [d.name for d in train_dir.iterdir() if d.is_dir()]

    for class_name in classes:
        class_files = list((train_dir / class_name).glob("*"))
        all_filepaths.extend(class_files)
        all_labels.extend([class_name] * len(class_files))


    if int(len(all_filepaths)) > num_samples:
        n_val = num_samples

        _, new_train_files, _, _ = train_test_split(
            all_filepaths, all_labels, 
            test_size=n_val, 
            stratify=all_labels, 
            random_state=seed
        )
    else:
        new_train_files = all_filepaths

    for file_path in new_train_files:
        dest_dir = new_train_dir / file_path.parent.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(str(file_path), str(dest_dir / file_path.name))
    
    
    print(f"Copied {len(new_train_files)} files to {new_train_dir}")


In [ ]:
for path in DataPath:
    select_subset_train(path, num_samples=1000, seed=0)

In [ ]:
def split_train(data_path: DataPath, val_ratio: float = 0.1, test_ratio: float = 0.1, seed: int = 0):
    """Moves a stratified subset of images from Train/ to a new Val/ folder and a new Test/ folder."""
    train_dir = Path(data_path) / "Train"
    val_dir = Path(data_path) / "Val"
    test_dir = Path(data_path) / "Test"

    if val_dir.exists():
        print(f"Validation folder already exists at {val_dir}. Skipping split.")
        return
    if test_dir.exists():
        print(f"Test folder already exists at {test_dir}. Skipping split.")
        return

    all_filepaths = []
    all_labels = []
    classes = [d.name for d in train_dir.iterdir() if d.is_dir()]

    for class_name in classes:
        class_files = list((train_dir / class_name).glob("*"))
        all_filepaths.extend(class_files)
        all_labels.extend([class_name] * len(class_files))

    n_val = int(len(all_filepaths) * val_ratio)
    n_test = int(len(all_filepaths) * test_ratio)

    _, val_files, _, _ = train_test_split(
        all_filepaths, all_labels, 
        test_size=n_val, 
        stratify=all_labels, 
        random_state=seed
    )

    for file_path in val_files:
        dest_dir = val_dir / file_path.parent.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.move(str(file_path), str(dest_dir / file_path.name))
        
        
    all_filepaths = []
    all_labels = []
    classes = [d.name for d in train_dir.iterdir() if d.is_dir()]

    for class_name in classes:
        class_files = list((train_dir / class_name).glob("*"))
        all_filepaths.extend(class_files)
        all_labels.extend([class_name] * len(class_files))
    
    
    _, test_files, _, _ = train_test_split(
        all_filepaths, all_labels, 
        test_size=n_test, 
        stratify=all_labels, 
        random_state=seed
    )

    for file_path in test_files:
        dest_dir = test_dir / file_path.parent.name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.move(str(file_path), str(dest_dir / file_path.name))
    
    
    print(f"Moved {len(val_files)} files to {val_dir} and {len(test_files)} files to {test_dir}")


In [ ]:
for path in DataPath:
    split_train(path)